In [10]:
import os
import json
import pandas as pd

# If running inside the notebooks/ folder, hop up to project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")


In [11]:
import pandas as pd

raw_data = {
    "well_name": ["Well-Alpha", "Well-Alpha", "Well-Alpha"],
    "core_no": [1, 2, 3],
    "depth_interval": ["3100 - 3115", "3115 - 3130.5", "3130.5 - 3148"],
    "recovery": ["92,5%", "100%", "88%"],
    "formation": ["Upper Agbada", "Upper Agbada", "Middle Agbada"]
}

df = pd.DataFrame(raw_data)
df

,well_name,core_no,depth_interval,recovery,formation
0,Well-Alpha,1,3100 - 3115,"92,5%",Upper Agbada
1,Well-Alpha,2,3115 - 3130.5,100%,Upper Agbada
2,Well-Alpha,3,3130.5 - 3148,88%,Middle Agbada


In [12]:
if "depth_interval" in df.columns:
        
    # Split the interval into two columns
    split_depths = df["depth_interval"].str.split(" - ", expand=True)

    # Convert them to float numbers
    df["top_depth_m"] = split_depths[0].astype(float)
    df["bottom_depth_m"] = split_depths[1].astype(float)

    # Drop the old glued columns
    df.drop(columns=["depth_interval"], inplace=True)

# View our progress
df

,well_name,core_no,recovery,formation,top_depth_m,bottom_depth_m
0,Well-Alpha,1,"92,5%",Upper Agbada,3100.0,3115.0
1,Well-Alpha,2,100%,Upper Agbada,3115.0,3130.5
2,Well-Alpha,3,88%,Middle Agbada,3130.5,3148.0


In [13]:
if "recovery" in df.columns:
    # 1. Replace European comma with dot
    df["recovery_pct"] = df["recovery"].str.replace(",", ".", regex=False)
    
    # 2. Remove the '%' symbol
    df["recovery_pct"] = df["recovery_pct"].str.replace("%", "", regex=False)
    
    # 3. Convert to float number
    df["recovery_pct"] = df["recovery_pct"].astype(float)
    
    # 4. Drop the old messy column
    df.drop(columns=["recovery"], inplace=True)

df


,well_name,core_no,formation,top_depth_m,bottom_depth_m,recovery_pct
0,Well-Alpha,1,Upper Agbada,3100.0,3115.0,92.5
1,Well-Alpha,2,Upper Agbada,3115.0,3130.5,100.0
2,Well-Alpha,3,Middle Agbada,3130.5,3148.0,88.0


In [ ]:
# Validate depths (bototm > top)
assert (df["bottom_depth_m"] > df["top_depth_m"]).all(), "Found a core where bottom depth <= top depth"

# Validate recovery is physically possible (0 to 100%)
assert ((df["recovery_pct"] >= 0) & (df["recovery_pct"] <= 100)).all(), "Error: found invalid recovery percentage"

print("All geological domain validation rules passed")

# Export deliverables to CSV
df.to_csv("data/clean_cores.csv", index=False)
print("Deliverables saved to clean_cores.csv")

All geological domain validation rules passed
Deliverables saved to clean_cores.csv


In [20]:
# 1. Define raw metadata records (1 record per well document)
raw_metadata = [
    {
        "well_name": "Well-Alpha",
        "licence": "OML-130",
        "field": "Egina",
        "operator": "TotalEnergies",
        "water_depth_m": 1550.0,
        "total_depth_m": 4500.0,
        "status": "Oil Producer"
    },
    {
        "well_name": "Well-Bravo",
        "licence": "OML-118",
        "field": "Bonga",
        "operator": "Shell",
        "water_depth_m": 1020.0,
        "total_depth_m": 4120.0,
        "status": "Appraisal"
    }
]

# 2. Convert to DataFrame
df_metadata = pd.DataFrame(raw_metadata)

# 3. Domain validation: Each well_name MUST be unique (1 record per well!)
assert df_metadata["well_name"].is_unique, "❌ Error: Duplicate well records found in metadata!"

# 4. Save to disk as Deliverable 2
df_metadata.to_csv("data/final_metadata.csv", index=False)

print("📁 Deliverable 2 saved successfully to 'final_metadata.csv'!")
df_metadata


📁 Deliverable 2 saved successfully to 'final_metadata.csv'!


,well_name,licence,field,operator,water_depth_m,total_depth_m,status
0,Well-Alpha,OML-130,Egina,TotalEnergies,1550.0,4500.0,Oil Producer
1,Well-Bravo,OML-118,Bonga,Shell,1020.0,4120.0,Appraisal


In [ ]:
# 1. Merge clean cores with well metadata
merged_df = pd.merge(df, df_metadata, on="well_name", how="left")

def create_core_chunks(df: pd.DataFrame) -> list[dict]:
    """
    Transform clean core rows into rich semantic chunks with metadata for ChromaDB
    """
    chunks = []
    
    for _, row in df.iterrows():
        # Calculate cored interval length
        length_m = round(row["bottom_depth_m"] - row["top_depth_m"], 1)
        
        # Rich natural sentence for semantic search
        sentence = (
            f"{row['well_name']} Core {row['core_no']} "
            f"(Field: {row['field']}, Operator: {row['operator']}, Licence: {row['licence']}) "
            f"was retrieved between {row['top_depth_m']}m and {row['bottom_depth_m']}m ({length_m}m interval) "
            f"in the {row['formation']} formation with a core recovery of {row['recovery_pct']}%. "
            f"Well status: {row['status']}."
        )
        
        # Structured metadata for exact filtering
        metadata = {
            "well_name": str(row["well_name"]),
            "core_no": int(row["core_no"]),
            "field": str(row["field"]),
            "operator": str(row["operator"]),
            "licence": str(row["licence"]),
            "formation": str(row["formation"]),
            "top_depth_m": float(row["top_depth_m"]),
            "bottom_depth_m": float(row["bottom_depth_m"]),
            "interval_length_m": float(length_m),
            "recovery_pct": float(row["recovery_pct"])
        }
        
        chunks.append({
            "chunk_id": f"core_{row['well_name']}_{row['core_no']}",
            "text": sentence,
            "metadata": metadata
        })
        
    return chunks

# 2. Execute on merged data
chunks = create_core_chunks(merged_df)

print(f"Successfully generated {len(chunks)} rich chunks!")
print(json.dumps(chunks[0], indent=2))


Successfully generated 3 rich chunks!
{
  "chunk_id": "core_Well-Alpha_1",
  "text": "Well-Alpha Core 1 (Field: Egina, Operator: TotalEnergies, Licence: OML-130) was retrieved between 3100.0m and 3115.0m (15.0m interval) in the Upper Agbada formation with a core recovery of 92.5%. Well status: Oil Producer.",
  "metadata": {
    "well_name": "Well-Alpha",
    "core_no": 1,
    "field": "Egina",
    "operator": "TotalEnergies",
    "licence": "OML-130",
    "formation": "Upper Agbada",
    "top_depth_m": 3100.0,
    "bottom_depth_m": 3115.0,
    "interval_length_m": 15.0,
    "recovery_pct": 92.5
  }
}


In [16]:
# Save our chunks to disk
file_name = "data/clean_chunks.json"
with open(file_name, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=4)
    
    
print(f"Chunk has been saved into {file_name}")

Chunk has been saved into data/clean_chunks.json


In [17]:
import chromadb

# Connect to local folder storage
client = chromadb.PersistentClient(path="./chroma_db")

# Get or create the collection
collection = client.get_or_create_collection(name="subsurface_cores")

# Unpack the data
ids = []
documents = []
metadatas = []

for c in chunks:
    ids.append(c["chunk_id"])
    documents.append(c["text"])
    metadatas.append(c["metadata"])
    
# Update and insert
collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print(f"Ingestion complete! Active vectors indexed in ChromaDB: {collection.count()}")


Ingestion complete! Active vectors indexed in ChromaDB: 3


In [18]:
# Define the question
query_text = "Which cores were recovered from the upper Agbada formation?"

# Query the vector store for top 2 closest matches
results = collection.query(
    query_texts=[query_text],
    n_results=2
)

for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0]), start=1):
    print(f"Rank {i}: ")
    print(f"Document: {doc}")
    print(f"  🏷️  Metadata: Well: {meta['well_name']} | Depth: {meta['top_depth_m']}m - {meta['bottom_depth_m']}m | Recovery: {meta['recovery_pct']}%\n")

    
# Print the matches clearly


Rank 1: 
Document: Well-Alpha Core 1 (Field: Egina, Operator: TotalEnergies, Licence: OML-130) was retrieved between 3100.0m and 3115.0m (15.0m interval) in the Upper Agbada formation with a core recovery of 92.5%. Well status: Oil Producer.
  🏷️  Metadata: Well: Well-Alpha | Depth: 3100.0m - 3115.0m | Recovery: 92.5%

Rank 2: 
Document: Well-Alpha Core 3 (Field: Egina, Operator: TotalEnergies, Licence: OML-130) was retrieved between 3130.5m and 3148.0m (17.5m interval) in the Middle Agbada formation with a core recovery of 88.0%. Well status: Oil Producer.
  🏷️  Metadata: Well: Well-Alpha | Depth: 3130.5m - 3148.0m | Recovery: 88.0%



In [19]:
# Ask a broad question
query_text = "core recovery details"

# Add mathematical filter: recovery_pct >= 90.0
filtered_results = collection.query(
    query_texts=[query_text],
    n_results=3,
    where={"recovery_pct": {"$gte": 90.0}}
)

# 3. Print the results
print(f"🔍 Query: \"{query_text}\" (Filter: recovery >= 90%)\n")
for i, (doc, meta) in enumerate(zip(filtered_results["documents"][0], filtered_results["metadatas"][0]), start=1):
    print(f"Rank {i}:")
    print(f"  📄 Document: {doc}")
    print(f"  🏷️  Recovery: {meta['recovery_pct']}% | Formation: {meta['formation']}\n")


🔍 Query: "core recovery details" (Filter: recovery >= 90%)

Rank 1:
  📄 Document: Well-Alpha Core 1 (Field: Egina, Operator: TotalEnergies, Licence: OML-130) was retrieved between 3100.0m and 3115.0m (15.0m interval) in the Upper Agbada formation with a core recovery of 92.5%. Well status: Oil Producer.
  🏷️  Recovery: 92.5% | Formation: Upper Agbada

Rank 2:
  📄 Document: Well-Alpha Core 2 (Field: Egina, Operator: TotalEnergies, Licence: OML-130) was retrieved between 3115.0m and 3130.5m (15.5m interval) in the Upper Agbada formation with a core recovery of 100.0%. Well status: Oil Producer.
  🏷️  Recovery: 100.0% | Formation: Upper Agbada



In [22]:
def format_retrieved_context(documents: list[str], metadatas: list[dict]) -> str:
    """
    Transforms raw ChromaDB search resutls into a clean, numbered context block for the LLM.
    """
    
    if not documents:
        return "No relevant subsurface records found."
    
    excerpts = []
    
    
    # pair each text with its metadta in a single pass.
    for i, (doc, meta) in enumerate(zip(documents, metadatas), start=1):
        header = f"[EXCERPT {i} | Well: {meta.get("well_name")} | Depth: {meta.get("top_depth_m")} - {meta.get("bottom_depth_m")}m]"
        excerpts.append(f"{header}\n\n{doc}")
        
    return "\n\n".join(excerpts)


context_block = format_retrieved_context(
    documents=filtered_results["documents"][0],
    metadatas=filtered_results["metadatas"][0]
)

print(context_block)

[EXCERPT 1 | Well: Well-Alpha | Depth: 3100.0 - 3115.0m]

Well-Alpha Core 1 (Field: Egina, Operator: TotalEnergies, Licence: OML-130) was retrieved between 3100.0m and 3115.0m (15.0m interval) in the Upper Agbada formation with a core recovery of 92.5%. Well status: Oil Producer.

[EXCERPT 2 | Well: Well-Alpha | Depth: 3115.0 - 3130.5m]

Well-Alpha Core 2 (Field: Egina, Operator: TotalEnergies, Licence: OML-130) was retrieved between 3115.0m and 3130.5m (15.5m interval) in the Upper Agbada formation with a core recovery of 100.0%. Well status: Oil Producer.


In [24]:
# 1. Define the Chief Geologist System Rules
SYSTEM_PROMPT = """You are a Chief Subsurface Geoscientist assisting an exploration team at the NAPE AICE hackathon.
Your duty is to answer technical questions using ONLY the provided subsurface archive excerpts.

CRITICAL RULES:
1. Base your answer strictly on the provided excerpts.
2. Cite the excerpt(s) you used to support each technical claim (e.g., "[EXCERPT 1]").
3. If the data is absent or insufficient to answer the question, state: 'Data not available in indexed well archives.'
4. Do NOT speculate, extrapolate, or invent reservoir properties under any circumstance.
5. Be concise, professional, and authoritative.
"""

def build_rag_prompt(question: str, context_block: str) -> tuple[str, str]:
    """
    Combines the question and context into (system_prompt, user_prompt)
    """
    user_prompt = f"""--- SUBSURFACE ARCHIVE EXCERPTS ---
{context_block}
----------------------------------

QUESTION: {question}

Provide a concise, grounded technical answer citing your sources:"""

    return SYSTEM_PROMPT, user_prompt

# Test it with a real question!
question = "Which cores in Well-Alpha had recovery of at least 90%, and at what depths?"
system_p, user_p = build_rag_prompt(question, context_block)

print("--- USER PROMPT PREVIEW ---")
print(user_p)


--- USER PROMPT PREVIEW ---
--- SUBSURFACE ARCHIVE EXCERPTS ---
[EXCERPT 1 | Well: Well-Alpha | Depth: 3100.0 - 3115.0m]

Well-Alpha Core 1 (Field: Egina, Operator: TotalEnergies, Licence: OML-130) was retrieved between 3100.0m and 3115.0m (15.0m interval) in the Upper Agbada formation with a core recovery of 92.5%. Well status: Oil Producer.

[EXCERPT 2 | Well: Well-Alpha | Depth: 3115.0 - 3130.5m]

Well-Alpha Core 2 (Field: Egina, Operator: TotalEnergies, Licence: OML-130) was retrieved between 3115.0m and 3130.5m (15.5m interval) in the Upper Agbada formation with a core recovery of 100.0%. Well status: Oil Producer.
----------------------------------

QUESTION: Which cores in Well-Alpha had recovery of at least 90%, and at what depths?

Provide a concise, grounded technical answer citing your sources:


In [27]:
import os
import requests
import json
import ollama

# -------------------------------------------------------------
# LLM Provider Functions
# -------------------------------------------------------------

def ask_ollama(user_prompt: str, system_prompt: str, model: str = "llama3.2:3b") -> str:
    """Runs the prompt through our local Ollama model (no internet needed)."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    response = ollama.chat(model=model, messages=messages, options={"temperature": 0.0})
    return response["message"]["content"]


def ask_gemini(user_prompt: str, system_prompt: str, api_key: str = None) -> str:
    """Calls Google Gemini using our API key."""
    key = api_key or os.getenv("GEMINI_API_KEY")
    if not key:
        return "Could not find a GEMINI_API_KEY."
    
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={key}"
    payload = {
        "contents": [{"parts": [{"text": user_prompt}]}],
        "systemInstruction": {"parts": [{"text": system_prompt}]}
    }
    res = requests.post(url, json=payload, headers={"Content-Type": "application/json"}, timeout=30)
    if res.status_code == 200:
        return res.json()["candidates"][0]["content"]["parts"][0]["text"].strip()
    return f"Gemini error ({res.status_code}): {res.text}"


def ask_openai(user_prompt: str, system_prompt: str, api_key: str = None) -> str:
    """Calls OpenAI using our API key."""
    key = api_key or os.getenv("OPENAI_API_KEY")
    if not key:
        return "Could not find an OPENAI_API_KEY."
    
    url = "https://api.openai.com/v1/chat/completions"
    headers = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
    payload = {
        "model": "gpt-4o-mini",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.0
    }
    res = requests.post(url, json=payload, headers=headers, timeout=30)
    if res.status_code == 200:
        return res.json()["choices"][0]["message"]["content"].strip()
    return f"OpenAI error ({res.status_code}): {res.text}"


# -------------------------------------------------------------
# Main Helper: Picks whichever service is available
# -------------------------------------------------------------

def ask_llm(user_prompt: str, system_prompt: str) -> str:
    """Checks for API keys first, and falls back to local Ollama if offline."""
    if os.getenv("GEMINI_API_KEY"):
        print("[+] Connecting to Google Gemini...")
        return ask_gemini(user_prompt, system_prompt)
        
    if os.getenv("OPENAI_API_KEY"):
        print("[+] Connecting to OpenAI...")
        return ask_openai(user_prompt, system_prompt)
        
    print("[+] Using local Ollama...")
    return ask_ollama(user_prompt, system_prompt)

print("AI connection functions ready.")


AI connection functions ready.


In [31]:
def ask_subsurface_ai(user_question: str, top_k: int = 2) -> str:
    """
    Takes any question -> searches ChromaDB -> builds fresh context -> calls LLM.
    """
    print(f"🔍 Searching ChromaDB for: \"{user_question}\"...")
    
    # 1. RETRIEVE: Search ChromaDB dynamically for THIS specific question!
    results = collection.query(
        query_texts=[user_question],
        n_results=top_k
    )
    
    # 2. AUGMENT: Format fresh context from whatever ChromaDB found
    fresh_context = format_retrieved_context(
        documents=results["documents"][0],
        metadatas=results["metadatas"][0]
    )
    
    # 3. PROMPT: Build prompt with our Chief Geologist rules
    sys_prompt, prompt_with_context = build_rag_prompt(user_question, fresh_context)
    
    # 4. GENERATE: Call the AI
    return ask_llm(user_prompt=prompt_with_context, system_prompt=sys_prompt)


In [32]:
# Test: The question that previously had no data!
answer = ask_subsurface_ai("What core was recovered from the Middle Agbada formation and what was its recovery?")

print("\nChief Geologist Report:\n")
print(answer)


🔍 Searching ChromaDB for: "What core was recovered from the Middle Agbada formation and what was its recovery?"...
[+] Connecting to Google Gemini...

Chief Geologist Report:

Well-Alpha Core 3 was recovered from the Middle Agbada formation with a core recovery of 88.0% [EXCERPT 2].


In [ ]:
import gradio as gr

def geologist_chat(message, history):
    """
    Passes the user's typed prompt into our subsurface RAG pipeline.
    """
    return ask_subsurface_ai(message)

# Build the interactive UI
with gr.Blocks(title="Ergon AI") as demo:
    gr.Markdown("# 🛢️ Ergon AI")
    gr.Markdown("Ask technical questions about well cores, formations, depth intervals, and recovery percentages.")
    
    with gr.Tab("💬 Geologist Chat"):
        gr.ChatInterface(
            fn=geologist_chat,
            examples=[
                "Which cores had recovery of at least 90%, and at what depths?",
                "What core was recovered from the Middle Agbada formation and what was its recovery?",
                "Show all core retrieval details for Well-Alpha."
            ]
        )
        
    with gr.Tab("📋 Indexed Cores Archive"):
        gr.Markdown("### Ground Truth Tabular Records")
        gr.DataFrame(df)

# Launch directly inside the notebook cell
demo.launch(inline=True)


c:\Users\Larry Mosh\hackathon-projects\ergon\team_ergon_ai\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


🔍 Searching ChromaDB for: "Which cores had recovery of at least 90%, and at what depths?"...
[+] Connecting to Google Gemini...
🔍 Searching ChromaDB for: "How're you doing?"...
[+] Connecting to Google Gemini...
🔍 Searching ChromaDB for: "What do you know about Total Energies?"...
[+] Connecting to Google Gemini...
